In [14]:
pip install requests deep-translator pandas


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [15]:
pip install nltk

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [23]:
import requests
import re
import pandas as pd
import concurrent.futures
from deep_translator import GoogleTranslator

# Funkcja pomocnicza przetwarzająca pojedynczy artykuł
def process_article(item):
    title = item.get("title", ["Brak tytułu"])[0]
    raw_abstract = item.get("abstract", "")
    
    clean_abstract = ""
    translated_abstract = ""
    
    if raw_abstract:
        # Oczyszczanie z tagów HTML/XML
        clean_abstract = re.sub(r'<[^<]+>', '', raw_abstract).strip()
        
        try:
            # Inicjalizacja tłumacza dla każdego wątku osobno
            translator = GoogleTranslator(source='auto', target='en')
            translated_abstract = translator.translate(clean_abstract)
        except Exception as e:
            translated_abstract = f"BŁĄD TŁUMACZENIA: {e}"
            
    return {
        "Tytuł": title,
        "Abstrakt_PL": clean_abstract,
        "Abstrakt_EN": translated_abstract
    }

def fetch_and_translate_fast(issn):
    url = f"https://api.crossref.org/journals/{issn}/works"
    
    all_items = []
    cursor = "*"  # Inicjalizacja kursora
    
    print(f"Rozpoczynam szybkie pobieranie danych dla ISSN: {issn}...")
    
    # 1. FAZA POBIERANIA (Sekwencyjna, ale przy użyciu maksymalnych paczek po 100)
    while True:
        params = {
            "filter": "has-abstract:true",
            "select": "title,abstract,is-referenced-by-count", # <-- ZMIANA TUTAJ
            "rows": 100,
            "cursor": cursor
        }
        
        response = requests.get(url, params=params)
        
        if response.status_code != 200:
            print(f"Błąd API Crossref: {response.status_code}")
            break
            
        data = response.json()
        items = data.get("message", {}).get("items", [])
        
        if not items:
            break  # Koniec wyników
            
        all_items.extend(items)
        print(f"Pobrano łącznie {len(all_items)} surowych rekordów...")
        
        # Pobranie nowego kursora do kolejnego zapytania
        next_cursor = data.get("message", {}).get("next-cursor")
        
        # Przerwanie pętli, jeśli kursor się nie zmienił lub nie istnieje
        if not next_cursor or next_cursor == cursor:
            break
            
        cursor = next_cursor

    print(f"\nPobieranie zakończone. Rozpoczynam współbieżne tłumaczenie {len(all_items)} abstraktów...")
    
    # 2. FAZA PRZETWARZANIA I TŁUMACZENIA (Współbieżna)
    results = []
    
    # Użycie ThreadPoolExecutor z 10 wątkami
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        # Funkcja map przetwarza listę zachowując jej oryginalną kolejność
        results = list(executor.map(process_article, all_items))

    print("Tłumaczenie zakończone.")
    
    # Konwersja do DataFrame
    return pd.DataFrame(results)

if __name__ == "__main__":
    JOURNAL_ISSN = "0239-6858" 
    df_edukacja = fetch_and_translate_fast(JOURNAL_ISSN)
    
    print("\nPodgląd wyników (5 pierwszych wierszy):")
    print(df_edukacja.head())

Rozpoczynam szybkie pobieranie danych dla ISSN: 0239-6858...
Pobrano łącznie 100 surowych rekordów...
Pobrano łącznie 125 surowych rekordów...

Pobieranie zakończone. Rozpoczynam współbieżne tłumaczenie 125 abstraktów...
Tłumaczenie zakończone.

Podgląd wyników (5 pierwszych wierszy):
                                               Tytuł  \
0  Narracja o wydajności i bezbłędności versus na...   
1  Fenomenografia jako klucz do świata dziecięcyc...   
2  Profil Rozwoju Emocjonalnego i Społecznego (PR...   
3  Nieobecności szkolne a szanse na edukacyjny su...   
4  Klasyczna paideia jako integralna uprawa człow...   

                                         Abstrakt_PL  \
0  The article concerns the ways in which learnin...   
1  Fenomenografia to strategia badawcza opracowan...   
2  Artykuł prezentuje możliwości zastosowania nar...   
3  This study examines the issue of school absent...   
4  Uprawa człowieka i jego człowieczeństwa należą...   

                                        

In [11]:
pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import requests
import re
import pandas as pd
import concurrent.futures
from deep_translator import GoogleTranslator
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Importy dla przetwarzania języka naturalnego
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Pobranie niezbędnych zasobów NLTK do lokalnego środowiska
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

def process_article(item):
    title = item.get("title", ["Brak tytułu"])[0]
    raw_abstract = item.get("abstract", "")
    citations = item.get("is-referenced-by-count", 0) # <-- NOWA LINIA
    
    clean_abstract = ""
    translated_abstract = ""
    
    if raw_abstract:
        clean_abstract = re.sub(r'<[^<]+>', '', raw_abstract).strip()
        try:
            translator = GoogleTranslator(source='auto', target='en')
            translated_abstract = translator.translate(clean_abstract)
        except Exception as e:
            translated_abstract = f"BŁĄD TŁUMACZENIA: {e}"
            
    return {
        "Tytuł": title,
        "Abstrakt_PL": clean_abstract,
        "Abstrakt_EN": translated_abstract,
        "Cytowania": citations # <-- NOWA LINIA
    }

def fetch_and_translate_fast(issn):
    url = f"https://api.crossref.org/journals/{issn}/works"
    all_items = []
    cursor = "*"  
    
    print(f"Rozpoczynam pobieranie danych dla ISSN: {issn}...")
    
    while True:
        params = {
            "filter": "has-abstract:true",
            "select": "title,abstract",
            "rows": 100,
            "cursor": cursor
        }
        
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print(f"Błąd API Crossref: {response.status_code}")
            break
            
        data = response.json()
        items = data.get("message", {}).get("items", [])
        
        if not items:
            break 
            
        all_items.extend(items)
        print(f"Pobrano {len(all_items)} surowych rekordów...")
        
        next_cursor = data.get("message", {}).get("next-cursor")
        if not next_cursor or next_cursor == cursor:
            break
        cursor = next_cursor

    print(f"\nRozpoczynam współbieżne tłumaczenie {len(all_items)} abstraktów...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        results = list(executor.map(process_article, all_items))

    print("Tłumaczenie zakończone.\n")
    return pd.DataFrame(results)

def preprocess_text(text):
    # Inicjalizacja narzędzi z NLTK
    stop_words = set(stopwords.words('english'))
    
    # Dodanie własnych słów wykluczonych
    custom_stop_words = {
        'article', 'paper', 'study', 'research', 'results', 'analysis', 
        'aim', 'purpose', 'data', 'method', 'methods', 'education', 
        'educational', 'school', 'schools', 'students', 'teachers', 
        'learning', 'also', 'may', 'used', 'based', 'two', 'one', 'new'
    }
    stop_words.update(custom_stop_words)
    
    # Tokenizacja i zamiana tekstu na małe litery
    words = word_tokenize(str(text).lower())
    
    # Filtrowanie
    cleaned_words = [
        word 
        for word in words 
        if word.isalpha() and word not in stop_words
    ]
    
    # Złączenie z powrotem w jeden ciąg znaków
    return " ".join(cleaned_words)

def perform_topic_modeling(df, n_topics=5, n_words=10):
    print(f"--- Rozpoczynam wyodrębnianie {n_topics} tematów (Topic Modeling) ---")
    
    df_valid = df[
        (df['Abstrakt_EN'] != "") & 
        (~df['Abstrakt_EN'].str.contains("BŁĄD TŁUMACZENIA", na=False))
    ].copy()
    
    if len(df_valid) < n_topics:
        print("Zbyt mało poprawnych danych do analizy.")
        return
        
    print("Trwa przetwarzanie tekstu (Tokenizacja, usuwanie rozszerzonych Stop Words)...")
    abstracts_processed = df_valid['Abstrakt_EN'].apply(preprocess_text).tolist()
    
    vectorizer = CountVectorizer(max_df=0.9, min_df=2)
    data_vectorized = vectorizer.fit_transform(abstracts_processed)
    
    lda_model = LatentDirichletAllocation(n_components=n_topics, random_state=42)
    lda_model.fit(data_vectorized)
    
    feature_names = vectorizer.get_feature_names_out()
    
    for topic_idx, topic in enumerate(lda_model.components_):
        top_features_ind = topic.argsort()[:-n_words - 1:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        
        print(f"\nTemat {topic_idx + 1}:")
        print(", ".join(top_features))
        
    print("\n--- Analiza zakończona ---")

if __name__ == "__main__":
    JOURNAL_ISSN = "0239-6858" 
    
    df_edukacja = fetch_and_translate_fast(JOURNAL_ISSN)
    
    if not df_edukacja.empty:
        perform_topic_modeling(df_edukacja, n_topics=5, n_words=10)

Rozpoczynam pobieranie danych dla ISSN: 0239-6858...
Pobrano 100 surowych rekordów...
Pobrano 125 surowych rekordów...

Rozpoczynam współbieżne tłumaczenie 125 abstraktów...


In [22]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

print("Analiza macierzy prawdopodobieństw, przypisywanie tematów i obliczanie cytowań...\n")

# 1. Przygotowanie poprawnych danych
df_valid = df_edukacja[
    (df_edukacja['Abstrakt_EN'] != "") & 
    (~df_edukacja['Abstrakt_EN'].str.contains("BŁĄD TŁUMACZENIA", na=False))
].copy()

# 2. Przetworzenie tekstów
abstracts_processed = df_valid['Abstrakt_EN'].apply(preprocess_text).tolist()

# 3. Wektoryzacja
vectorizer = CountVectorizer(max_df=0.9, min_df=2)
data_vectorized = vectorizer.fit_transform(abstracts_processed)

# 4. Inicjalizacja i trenowanie modelu LDA
lda_model = LatentDirichletAllocation(n_components=5, random_state=42)
document_topic_matrix = lda_model.fit_transform(data_vectorized)

# 5. Wybór dominującego tematu dla każdego artykułu
df_valid['Przypisany_Temat'] = np.argmax(document_topic_matrix, axis=1) + 1

# 6. Grupowanie danych, zliczanie artykułów oraz obliczanie mediany cytowań
# Metoda agg() pozwala na zdefiniowanie wielu agregacji jednocześnie
statystyki_tematow = df_valid.groupby('Przypisany_Temat').agg(
    Liczba_Artykułów=('Przypisany_Temat', 'size'),
    Mediana_Cytowań=('Cytowania', 'median')
).reset_index()

# Zmiana nazwy kolumny klucza głównego dla lepszej czytelności
statystyki_tematow.rename(columns={'Przypisany_Temat': 'Numer Tematu'}, inplace=True)

print("--- Podsumowanie artykułów i cytowań w poszczególnych tematach ---")
print(statystyki_tematow.to_string(index=False))

Analiza macierzy prawdopodobieństw, przypisywanie tematów i obliczanie cytowań...



KeyError: "Column(s) ['Cytowania'] do not exist"